In [1]:
# Imports
import polars as pl
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
FIGURE_DIR = PROJECT_ROOT / "figures"
SUBMISSION_DIR = PROJECT_ROOT / "results" / "submissions"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "dat_train1.csv"
CLEAN_PATH = DATA_DIR / "dat_train1_clean.csv"

# lazy load
df = pl.scan_csv(DATA_PATH)


In [3]:
# TASK 1 & 2 -- data cleaning

#1.1
num_rows = df.select(pl.len()).collect().item()
#1.2
num_unique_ids = df.select(pl.col("id").n_unique()).collect().item()

#1.3
earliest, latest = df.select([
    pl.col("event_timestamp").min().alias("earliest"),
    pl.col("event_timestamp").max().alias("latest")
]).collect().row(0)



print("Task 1.1 - Number of rows:", num_rows)

print("Task 1.2 - Number of unique IDs:", num_unique_ids)

print("Task 1.3 - Earliest timestamp:", earliest)
print("Task 1.3 - Latest timestamp:", latest)
## task 2

#2.1
total = num_rows

unique = df.unique(
    subset=["id", "event_name", "event_timestamp"]
).select(pl.len()).collect().item()

duplicates = total - unique
proportion = duplicates / total

# remove duplicates
df_clean = df.unique(subset=["id", "event_name", "event_timestamp"])

# fix the action counter
df_clean = df_clean.sort(["id", "event_timestamp"]).with_columns(
    pl.int_range(1, pl.len() + 1).over("id").alias("journey_steps_until_end")
)


df_clean.collect().write_csv(CLEAN_PATH)
df_clean = pl.scan_csv(CLEAN_PATH)

num_rows_clean = df_clean.select(pl.len()).collect().item()

print("Task 2.1 - Number of duplicates:", duplicates)
print("Task 2.1 - Proportion of duplicates:", proportion)

print("Task 2.2 - Rows after removing duplicates:", num_rows_clean)


Task 1.1 - Number of rows: 54960961
Task 1.2 - Number of unique IDs: 1430445
Task 1.3 - Earliest timestamp: 2020-11-03T03:31:30Z
Task 1.3 - Latest timestamp: 2023-01-23T12:29:56Z
Task 2.1 - Number of duplicates: 3112100
Task 2.1 - Proportion of duplicates: 0.056623827956720045
Task 2.2 - Rows after removing duplicates: 51848861


In [4]:
# TASK 3 -- statistics 

# sample
unique_ids = df.select("id").unique().collect()
# sample 100k ids
sample_ids = unique_ids.sample(n=100000, shuffle=True)
# filter full df by sampled ids
df_sample = df.filter(pl.col("id").is_in(sample_ids["id"])).collect()

# parse timestamp once
df_sample = df_sample.with_columns(
    pl.col("event_timestamp").str.strptime(
        pl.Datetime,
        format="%Y-%m-%dT%H:%M:%SZ"
    )
)

# # 3.1
actions_per_journey = df_sample.group_by("id").agg(
    pl.len().alias("num_actions")
)

journey_time = df_sample.group_by("id").agg(
    (pl.col("event_timestamp").max() - pl.col("event_timestamp").min()).alias("duration")
)
journey_time_seconds = journey_time.with_columns(
    pl.col("duration").dt.total_seconds().alias("duration_sec")
)

# 3.2
common_actions = df_sample.group_by("event_name").agg(
    pl.len().alias("count")
).sort("count", descending=True)

# 3.3
time_between = df_sample.sort(["id", "event_timestamp"]).with_columns(
    (pl.col("event_timestamp") - pl.col("event_timestamp").shift(1))
    .over("id")
    .alias("time_diff")
)
time_between_seconds = time_between.with_columns(
    pl.col("time_diff").dt.total_seconds().alias("time_diff_sec")
)

/var/folders/q7/zr7bldv11l9_4tx8bpbbfm4m0000gn/T/ipykernel_49451/2847510455.py:8: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_sample = df.filter(pl.col("id").is_in(sample_ids["id"])).collect()


In [5]:
# TASK 4 -- flattening

def flatten_journeys_parquet(input_csv_path, output_parquet_path):
    """
    Removes duplicates and flattens an event csv file so each journey (id)
    is one row, with the ordered journey stored as structs plus summary features.
    """
    q = pl.scan_csv(input_csv_path)
    q = q.unique(subset=["id", "event_name", "event_timestamp"])
    q = q.with_columns(
        pl.col("event_timestamp").str.to_datetime(time_zone="UTC")
    )
    q = q.sort(["id", "event_timestamp", "event_name"])

    df_flat = (
        q.group_by("id")
        .agg([
            pl.struct(["event_timestamp", "event_name"]).alias("journey"),
            pl.len().alias("num_actions"),
            pl.col("event_name").n_unique().alias("num_unique_actions"),
            pl.col("event_timestamp").min().alias("start_time"),
            pl.col("event_timestamp").max().alias("end_time"),
            (
                pl.col("event_timestamp").max() - pl.col("event_timestamp").min()
            ).dt.total_seconds().alias("duration_seconds"),
            pl.col("event_name").first().alias("first_action"),
            pl.col("event_name").last().alias("last_action"),
        ])
    )

    df_flat.sink_parquet(output_parquet_path)


OUTPUT_PATH = DATA_DIR / "journeys_flattened.parquet"

flatten_journeys_parquet(DATA_PATH, OUTPUT_PATH)

df_flat = pl.read_parquet(OUTPUT_PATH)
print("Task 4 - Flattened journey dataset preview:")
print(df_flat.head())

Task 4 - Flattened journey dataset preview:
shape: (5, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ journey   ┆ num_actio ┆ num_uniqu ┆ … ┆ end_time  ┆ duration_ ┆ first_act ┆ last_act │
│ ---       ┆ ---       ┆ ns        ┆ e_actions ┆   ┆ ---       ┆ seconds   ┆ ion       ┆ ion      │
│ str       ┆ list[stru ┆ ---       ┆ ---       ┆   ┆ datetime[ ┆ ---       ┆ ---       ┆ ---      │
│           ┆ ct[2]]    ┆ u32       ┆ u32       ┆   ┆ μs, UTC]  ┆ i64       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ -10000012 ┆ [{2022-10 ┆ 7         ┆ 4         ┆ … ┆ 2022-12-0 ┆ 2868448   ┆ applicati ┆ browse_p │
│ 71        ┆ -31       ┆           ┆           ┆   ┆ 3         ┆           ┆ on_web_ap ┆ roducts  │
│ 551641434 ┆ 13:45:59  ┆           ┆           ┆   ┆ 18:33:27  ┆           ┆ proved    ┆          │
│           ┆ UTC,"app… ┆        

In [6]:
# TASK 5

import matplotlib.pyplot as plt
import pandas as pd

FLATTENED_PATH = DATA_DIR / "journeys_flattened.parquet"
LABELED_PATH = DATA_DIR / "journeys_labeled.parquet"

ORDER_SHIPPED_ACTION = "order_shipped"

journeys = pl.read_parquet(FLATTENED_PATH)

dataset_end_time = (
    pl.scan_csv(DATA_PATH)
    .with_columns(
        pl.col("event_timestamp").str.to_datetime(time_zone="UTC")
    )
    .select(pl.col("event_timestamp").max().alias("dataset_end_time"))
    .collect()
    .item()
)

journeys_labeled = journeys.with_columns([
    (pl.col("last_action") == ORDER_SHIPPED_ACTION).alias("is_successful"),
    (
        (pl.lit(dataset_end_time) - pl.col("end_time")).dt.total_days() >= 60
    ).alias("inactive_60_days"),
]).with_columns([
    (
        pl.when(pl.col("is_successful"))
        .then(pl.lit("successful"))
        .when((~pl.col("is_successful")) & pl.col("inactive_60_days"))
        .then(pl.lit("incomplete"))
        .otherwise(pl.lit("other"))
    ).alias("journey_status")
])

In [7]:
# TASK 6
# Create training data for predictive modeling using Option A:
# one snapshot per journey

import random
import polars as pl

# ---- PATHS ----
TRAIN_PARQUET_PATH = DATA_DIR / "journey_training_optionA.parquet"
TRAIN_CSV_PATH = DATA_DIR / "journey_training_optionA.csv"

# We assume journeys_labeled already exists from Task 5 and contains:
# - id
# - journey (list of structs with event_timestamp and event_name)
# - journey_status in {"successful", "incomplete", "other"}

# --------------------------------------------------
# 6.1 Keep only journeys with clear final outcomes
# --------------------------------------------------
model_base = journeys_labeled.filter(
    pl.col("journey_status").is_in(["successful", "incomplete"])
)

print("\nTask 6.1 - Number of labeled journeys kept for modeling:")
print(
    model_base.group_by("journey_status")
    .agg(pl.len().alias("count"))
    .sort("journey_status")
)

# --------------------------------------------------
# 6.2 Create one partial snapshot per journey
# --------------------------------------------------
def make_snapshot(row):
    journey = row["journey"]
    n = len(journey)

    if n <= 1:
        k = 1
    else:
        k = random.randint(1, n - 1)

    prefix = journey[:k]

    timestamps = [step["event_timestamp"] for step in prefix]
    actions = [step["event_name"] for step in prefix]

    start_time = timestamps[0]
    current_time = timestamps[-1]
    current_last_action = actions[-1]
    first_action = actions[0]

    prefix_duration_seconds = int((current_time - start_time).total_seconds()) if k > 1 else 0
    num_actions_so_far = k
    num_unique_actions_so_far = len(set(actions))

    if k > 1:
        avg_gap_seconds = prefix_duration_seconds / (k - 1)
        time_since_prev_action_seconds = int((timestamps[-1] - timestamps[-2]).total_seconds())
    else:
        avg_gap_seconds = 0
        time_since_prev_action_seconds = 0

    return {
        "id": row["id"],
        "label": 1 if row["journey_status"] == "successful" else 0,
        "final_outcome": row["journey_status"],

        "full_num_actions": n,
        "snapshot_num_actions": num_actions_so_far,
        "snapshot_frac_of_journey": num_actions_so_far / n,

        "first_action_so_far": first_action,
        "current_last_action": current_last_action,
        "num_unique_actions_so_far": num_unique_actions_so_far,
        "prefix_duration_seconds": prefix_duration_seconds,
        "avg_gap_seconds": avg_gap_seconds,
        "time_since_prev_action_seconds": time_since_prev_action_seconds,

        "prefix_actions": actions,
    }

snapshot_rows = [make_snapshot(row) for row in model_base.iter_rows(named=True)]
training_df = pl.DataFrame(snapshot_rows)

print("\nTask 6.2 - Training dataset preview:")
print(training_df.head())

# --------------------------------------------------
# 6.3 Add count features for common actions
# --------------------------------------------------
top_actions = (
    training_df.explode("prefix_actions")
    .group_by("prefix_actions")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(15)
    .get_column("prefix_actions")
    .to_list()
)

for action_name in top_actions:
    safe_name = str(action_name).replace(" ", "_").replace("/", "_")
    training_df = training_df.with_columns(
        pl.col("prefix_actions")
        .list.eval(pl.element() == action_name)
        .list.sum()
        .alias(f"action_count_{safe_name}")
    )

print("\nTask 6.3 - Added action count features for top actions:")
print(top_actions)

# --------------------------------------------------
# 6.4 Save modeling dataset
# --------------------------------------------------
training_df.write_parquet(TRAIN_PARQUET_PATH)

training_df_csv = training_df.drop("prefix_actions")
training_df_csv.write_csv(TRAIN_CSV_PATH)

print("\nSaved training data to:")
print(TRAIN_PARQUET_PATH)
print(TRAIN_CSV_PATH)

# --------------------------------------------------
# 6.5 Basic label balance check
# --------------------------------------------------
print("\nTask 6.5 - Label balance:")
print(
    training_df.group_by(["label", "final_outcome"])
    .agg(pl.len().alias("count"))
    .sort("label")
)


Task 6.1 - Number of labeled journeys kept for modeling:
shape: (2, 2)
┌────────────────┬────────┐
│ journey_status ┆ count  │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ incomplete     ┆ 992757 │
│ successful     ┆ 279363 │
└────────────────┴────────┘

Task 6.2 - Training dataset preview:
shape: (5, 13)
┌────────────┬───────┬────────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ label ┆ final_outc ┆ full_num_a ┆ … ┆ prefix_du ┆ avg_gap_s ┆ time_sinc ┆ prefix_ac │
│ ---        ┆ ---   ┆ ome        ┆ ctions     ┆   ┆ ration_se ┆ econds    ┆ e_prev_ac ┆ tions     │
│ str        ┆ i64   ┆ ---        ┆ ---        ┆   ┆ conds     ┆ ---       ┆ tion_seco ┆ ---       │
│            ┆       ┆ str        ┆ i64        ┆   ┆ ---       ┆ f64       ┆ nds       ┆ list[str] │
│            ┆       ┆            ┆            ┆   ┆ i64       ┆           ┆ ---       ┆           │
│            ┆       ┆            ┆          

In [2]:
# TASK 7
# Fit a simple Random Forest model

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# ---- PATHS ----
TRAIN_PARQUET_PATH = DATA_DIR / "journey_training_optionA.parquet"
FIG11_PATH = FIGURE_DIR / "fig11_feature_importance.png"
SUBMISSION_PATH = SUBMISSION_DIR / "kaggle_submission.csv"

# --------------------------------------------------
# 7.1 Load training data
# --------------------------------------------------
df = pl.read_parquet(TRAIN_PARQUET_PATH)

df_model = df.drop(["prefix_actions", "final_outcome"])
df_pd = df_model.to_pandas()

print("\nTask 7.1 - Training data shape:")
print(df_pd.shape)

# --------------------------------------------------
# 7.2 Encode categorical variables
# --------------------------------------------------
cat_cols = ["first_action_so_far", "current_last_action"]

for col in cat_cols:
    le = LabelEncoder()
    df_pd[col] = le.fit_transform(df_pd[col])

# --------------------------------------------------
# 7.3 Define features + target
# --------------------------------------------------
X = df_pd.drop(columns=["label", "id"])
y = df_pd["label"]

# --------------------------------------------------
# 7.4 Fit Random Forest
# --------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    oob_score=True,
    n_jobs=-1
)

rf.fit(X, y)

print("\nTask 7.2 - OOB Accuracy:")
print(rf.oob_score_)

# --------------------------------------------------
# 7.5 Feature importance
# --------------------------------------------------
importances = rf.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\nTop features:")
print(importance_df.head(10))

# --------------------------------------------------
# 7.6 Plot feature importance
# --------------------------------------------------
top_k = 10
top_features = importance_df.head(top_k)

plt.barh(top_features["feature"], top_features["importance"])
plt.gca().invert_yaxis()
plt.title("Top Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(FIG11_PATH)
# plt.show()
plt.clf()


# ---------------- TASK 8 ----------------

train_probs = rf.predict_proba(X)[:, 1]

submission = pd.DataFrame({
    "id": df_pd["id"],
    "order_shipped": train_probs
})

submission.to_csv(SUBMISSION_PATH, index=False)

print("\nSaved submission file to:", SUBMISSION_PATH)
print(submission.head())
print(submission.shape)

print("\nPrediction summary:")
print(submission["order_shipped"].describe())


Task 7.1 - Training data shape:
(1272120, 26)

Task 7.2 - OOB Accuracy:
0.8470891110901487

Top features:
                              feature  importance
20  action_count_account_activitation    0.129603
0                    full_num_actions    0.106572
8      time_since_prev_action_seconds    0.091026
7                     avg_gap_seconds    0.090946
6             prefix_duration_seconds    0.089740
2            snapshot_frac_of_journey    0.087428
9        action_count_browse_products    0.042744
22     action_count_place_downpayment    0.041389
1                snapshot_num_actions    0.039645
3                 first_action_so_far    0.034861

Saved submission file to: data/kaggle_submission.csv
                        id  order_shipped
0      -100000293 92584960       0.000000
1   -100001025 -1674572031       0.060000
2  -1000011207 -1448136887       0.264214
3    -100001165 1418179836       0.000000
4  -1000012940 -1479669207       0.100000
(1272120, 2)

Prediction summary:
cou

<Figure size 640x480 with 0 Axes>

In [8]:
print("Trained trees:", len(rf.estimators_))
print("Classes:", rf.classes_)

Trained trees: 100
Classes: [0 1]


In [3]:
# ---- PATHS ----
TEST_PATH = DATA_DIR / "open_journeys1.csv"
SUBMISSION_PATH = SUBMISSION_DIR / "kaggle_submission.csv"

# --------------------------------------------------
# LOAD TEST DATA
# --------------------------------------------------
df_test = pl.read_csv(TEST_PATH)

print(df_test.head())
print(df_test.columns)

# parse timestamp
df_test = df_test.with_columns(
    pl.col("event_timestamp").str.to_datetime(time_zone="UTC")
)

# --------------------------------------------------
# GROUP INTO JOURNEYS (same idea as training)
# --------------------------------------------------
test_journeys = (
    df_test.sort(["id", "event_timestamp"])
    .group_by("id")
    .agg([
        pl.col("event_name").alias("journey"),   # FIXED (was ed_id)
        pl.col("event_timestamp").alias("timestamps"),
        pl.col("event_name").last().alias("last_action"),
        pl.len().alias("num_actions"),
    ])
)

print(test_journeys.head())


# --------------------------------------------------
# FEATURE ENGINEERING (same as Task 6)
# --------------------------------------------------
import pandas as pd

def make_test_features(row):
    journey = row["journey"]
    timestamps = row["timestamps"]

    n = len(journey)

    if n > 1:
        prefix_duration = int((timestamps[-1] - timestamps[0]).total_seconds())
        avg_gap = prefix_duration / (n - 1)
        time_since_prev = int((timestamps[-1] - timestamps[-2]).total_seconds())
    else:
        prefix_duration = 0
        avg_gap = 0
        time_since_prev = 0

    return {
        "id": row["id"],

        "full_num_actions": n,
        "snapshot_num_actions": n,
        "snapshot_frac_of_journey": 1.0,

        "first_action_so_far": journey[0],
        "current_last_action": journey[-1],
        "num_unique_actions_so_far": len(set(journey)),

        "prefix_duration_seconds": prefix_duration,
        "avg_gap_seconds": avg_gap,
        "time_since_prev_action_seconds": time_since_prev,
    }

test_df = pd.DataFrame(
    [make_test_features(r) for r in test_journeys.iter_rows(named=True)]
)

print(test_df.head())


# --------------------------------------------------
# ENCODE CATEGORICAL VARIABLES (same as training)
# --------------------------------------------------
from sklearn.preprocessing import LabelEncoder

test_df_enc = test_df.copy()
encoders = {}

train_df = df_pd.copy()  # from Task 7

# fit encoders on TRAIN data
for col in ["first_action_so_far", "current_last_action"]:
    le = LabelEncoder()
    le.fit(train_df[col])
    encoders[col] = le

# apply to TEST data
for col in ["first_action_so_far", "current_last_action"]:
    le = encoders[col]
    test_df_enc[col] = test_df_enc[col].map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )


# --------------------------------------------------
# ALIGN FEATURES WITH TRAINING
# --------------------------------------------------
for col in X.columns:
    if col not in test_df_enc.columns:
        test_df_enc[col] = 0

X_test = test_df_enc[X.columns]


# --------------------------------------------------
# PREDICT
# --------------------------------------------------
preds = rf.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": test_df["id"],
    "order_shipped": preds
})

submission.to_csv(SUBMISSION_PATH, index=False)

print("\nSaved submission to:", SUBMISSION_PATH)
print(submission.head())

shape: (5, 8)
┌─────────────┬────────────┬───────┬──────────────┬──────────────┬─────────────┬─────────────┬─────┐
│ customer_id ┆ account_id ┆ ed_id ┆ event_name   ┆ event_timest ┆ journey_ste ┆ id          ┆ sep │
│ ---         ┆ ---        ┆ ---   ┆ ---          ┆ amp          ┆ ps_until_en ┆ ---         ┆ --- │
│ i64         ┆ i64        ┆ i64   ┆ str          ┆ ---          ┆ d           ┆ str         ┆ str │
│             ┆            ┆       ┆              ┆ str          ┆ ---         ┆             ┆     │
│             ┆            ┆       ┆              ┆              ┆ i64         ┆             ┆     │
╞═════════════╪════════════╪═══════╪══════════════╪══════════════╪═════════════╪═════════════╪═════╡
│ -1522106248 ┆ -693958153 ┆ 19    ┆ application_ ┆ 2022-12-19T2 ┆ 1           ┆ -1522106248 ┆ -   │
│             ┆            ┆       ┆ web_view     ┆ 3:05:56Z     ┆             ┆ -693958153  ┆     │
│ -1522106248 ┆ -693958153 ┆ 19    ┆ application_ ┆ 2022-12-19T2 ┆ 2         

In [10]:
# TASK 8 -- getting the kaggle file

import pandas as pd

# ---- PATHS ----
SAMPLE_PATH = DATA_DIR / "open_journeys1_flattened_all0.csv"
FINAL_SUBMISSION_PATH = SUBMISSION_DIR / "kaggle_submission_final.csv"

# --- STEP 1: Ensure predictions already exist ---
submission = pd.DataFrame({
    "id": test_df["id"],
    "order_shipped": preds
})

# --- STEP 2: Match sample submission ordering ---
sample = pd.read_csv(SAMPLE_PATH)

submission = sample[["id"]].merge(submission, on="id", how="left")

# Fill any missing predictions
submission["order_shipped"] = submission["order_shipped"].fillna(0)

# --- STEP 3: Save file ---
submission.to_csv(FINAL_SUBMISSION_PATH, index=False)

# --- STEP 4: Quick sanity check ---
print("Saved to:", FINAL_SUBMISSION_PATH)
print(submission.head())
print(submission.shape)
print(submission["order_shipped"].describe())

Saved to: data/kaggle_submission_final.csv
                        id  order_shipped
0    -1000001271 551641434       0.417106
1   -100001164 -1710062169       0.016437
2    -1000073039 494887319       0.370075
3  -1000092799 -1963858498       0.355063
4    -100009516 1394046265       0.497106
(158325, 2)
count    158325.000000
mean          0.370379
std           0.117758
min           0.016437
25%           0.345063
50%           0.410075
75%           0.447106
max           0.600075
Name: order_shipped, dtype: float64


In [ ]:
# TASK 9 -- SHAP for the Random Forest (fast ~1 minute)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import shap
except ImportError as e:
    raise ImportError(
        "Missing dependency `shap`. Install it (e.g. `pip install shap`) and re-run this cell."
    ) from e

# Assumes Task 7 already created:
# - rf (trained RandomForestClassifier)
# - X  (pandas DataFrame of features)

# Speed knobs:
# - Keep N_SHAP small
# - Use TreeExplainer approximate mode
# - Skip beeswarm (slow) and just do a bar importance plot
N_SHAP = 20

X_shap = X.sample(n=min(N_SHAP, len(X)), random_state=42).copy()

explainer = shap.TreeExplainer(
    rf,
    feature_perturbation="tree_path_dependent",
)

# `approximate=True` is much faster for tree ensembles
shap_values_raw = explainer.shap_values(
    X_shap,
    check_additivity=False,
    approximate=True,
)

# Binary classification handling
if isinstance(shap_values_raw, list):
    shap_values = shap_values_raw[1]
else:
    shap_values = shap_values_raw

# Some SHAP versions return shape (n_samples, n_features, n_classes)
if hasattr(shap_values, "ndim") and shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

# ---- Global importance bar plot (fast) ----
FIG13_PATH = FIGURE_DIR / "fig13_shap_global_importance_fast.png"

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False)

top_k = 15
_top = shap_importance_df.head(top_k).sort_values("mean_abs_shap", ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(_top["feature"], _top["mean_abs_shap"])
plt.xlabel("Mean absolute SHAP value")
plt.title("Global Variable Importance (SHAP, fast approx)")
plt.tight_layout()
plt.savefig(FIG13_PATH, dpi=250, bbox_inches="tight")
plt.clf()

print("Saved:", FIG13_PATH)
print("Top SHAP features:")
print(shap_importance_df.head(10))

# ---- Persist to disk (optional) ----
SHAP_VALUES_PATH = DATA_DIR / "rf_shap_values_fast.npy"
SHAP_FEATURES_PATH = DATA_DIR / "rf_shap_features_fast.csv"

np.save(SHAP_VALUES_PATH, shap_values)
X_shap.to_csv(SHAP_FEATURES_PATH, index=False)

print("Saved:", SHAP_VALUES_PATH)
print("Saved:", SHAP_FEATURES_PATH)
print("shap_values shape:", shap_values.shape)
print("X_shap shape:", X_shap.shape)


In [ ]:
# Beeswarm plot from saved SHAP values (fast sample)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

SHAP_VALUES_PATH = DATA_DIR / "rf_shap_values_fast.npy"
SHAP_FEATURES_PATH = DATA_DIR / "rf_shap_features_fast.csv"
FIG12_PATH = FIGURE_DIR / "fig12_shap_beeswarm_fast.png"

shap_values = np.load(SHAP_VALUES_PATH)
X_shap = pd.read_csv(SHAP_FEATURES_PATH)

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="dot",
    max_display=15,
    show=False,
)
plt.title("SHAP Beeswarm Plot (fast sample)", pad=12)
plt.tight_layout()
plt.savefig(FIG12_PATH, dpi=200, bbox_inches="tight")
plt.clf()

print("Saved:", FIG12_PATH)
print("Loaded shap_values:", shap_values.shape)
print("Loaded X_shap:", X_shap.shape)
